# TP 3 — Apprentissage non supervisé (MLNS)
### Partie III de la conférence · Licence 1 → Master 2

> **TechForge Club · FST — Université de Nouakchott**  
> Conférence **« L’intelligence artificielle : au-delà de la théorie »** · الذكاء الاصطناعي : ما وراء النظرية  
> Animée par **Dr El Benany Med Mahmoud** · samedi 14 novembre 2026 · 12 h 30 · Amphi 5 – FST

**Trois parcours dans le même notebook — choisissez le vôtre :**

| Repère | Public conseillé | Ce que l’on vous demande |
|:---:|---|---|
| 🟢 **Découverte** | Licence 1 – Licence 2 | Exécuter, observer, modifier une valeur, répondre en une phrase. Aucune démonstration exigée. |
| 🟡 **Approfondissement** | Licence 3 | Refaire les calculs à la main, compléter du code, interpréter les résultats. |
| 🔴 **Expert** | Master 1 – Master 2 | Démontrer, implémenter sans bibliothèque, critiquer le protocole, mener un mini-projet. |

Chaque exercice **À faire** porte un repère. Un étudiant de licence peut s’arrêter aux 🟢 d’une section et passer à la suivante ; un étudiant de master traite les trois niveaux. Les cellules `verifier(...)` contrôlent votre réponse sans afficher la solution.

**Objectifs.** Découvrir des structures dans des données **sans utiliser d’étiquettes pendant l’apprentissage** : distances et normalisation, k-means calculé à la main puis programmé, choix de $k$, DBSCAN, classification hiérarchique, mélanges gaussiens, ACP, t-SNE, anomalies, règles d’association, NMF — et surtout savoir en interpréter les limites.

**Diapositives associées :** Partie III · 19 à 36.

**Prérequis.** 🟢 exécuter une cellule, lire un nuage de points ; 🟡 Python, tableaux NumPy, moyenne et variance ; 🔴 algèbre linéaire (valeurs propres), probabilités.

Les jeux de données intégrés à scikit-learn fonctionnent **hors ligne**.

> **Règle du TP.** Des classes connues peuvent être utilisées *après* le clustering, uniquement pour une évaluation externe. Elles ne doivent jamais guider l’apprentissage non supervisé.

| Section | Thème | Diapos | 🟢 | 🟡 | 🔴 | Durée |
|---|---|:---:|:---:|:---:|:---:|:---:|
| 1 | Données sans étiquettes, distances, normalisation | 19–20 | ✔ | ✔ | ✔ | 20 min |
| 2 | k-means : calcul manuel, itérations de Lloyd | 21–22 | ✔ | ✔ | ✔ | 30 min |
| 3 | Choisir $k$ : coude et silhouette | 23 | ✔ | ✔ | ✔ | 15 min |
| 4 | DBSCAN | 24 | ✔ | ✔ | ✔ | 15 min |
| 5 | Classification hiérarchique, dendrogramme | 25–26 |  | ✔ | ✔ | 15 min |
| 6 | Mélanges gaussiens | 27 |  | ✔ | ✔ | 15 min |
| 7 | ACP | 28–30 | ✔ | ✔ | ✔ | 30 min |
| 8 | t-SNE | 31 | ✔ | ✔ | ✔ | 10 min |
| 9–11 | Anomalies, règles d’association, NMF | 32–34 | ✔ | ✔ | ✔ | 30 min |
| 12 | Évaluer sans étiquettes, stabilité | 35 |  | ✔ | ✔ | 15 min |
| 13–16 | Défis, mini-projets, barème, autoévaluation | 36 | ✔ | ✔ | ✔ | projet |

In [ ]:
# Installation (à décommenter une seule fois si nécessaire) :
# %pip install numpy pandas matplotlib scikit-learn scipy ipywidgets
# --- Outils communs aux quatre TP de la conférence ---
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
SEED = 42
rng = np.random.default_rng(SEED)
plt.rcParams.update({'figure.figsize': (7, 4), 'axes.grid': True, 'grid.alpha': .3})

def verifier(nom, reponse, attendu, tol=1e-2):
    """Compare votre réponse à la valeur attendue, sans l'afficher."""
    if reponse is None:
        print(f'⏳ {nom} : à compléter (remplacez None par votre réponse)'); return
    ok = np.allclose(np.asarray(reponse, dtype=float), np.asarray(attendu, dtype=float), atol=tol)
    print(f'✅ {nom} : correct' if ok else f'❌ {nom} : à revoir — reprenez le calcul')

def explorer(f, **curseurs):
    """Curseurs interactifs si ipywidgets est installé ; sinon exécution avec les valeurs par défaut.
    Chaque curseur est donné sous la forme nom=(min, max, pas, valeur_par_defaut)."""
    try:
        import ipywidgets as w
        ctl = {k: (w.FloatSlider if any(isinstance(x, float) for x in v) else w.IntSlider)(
                   min=v[0], max=v[1], step=v[2], value=v[3], description=k, continuous_update=False)
               for k, v in curseurs.items()}
        w.interact(f, **ctl)
    except ImportError:
        print('(ipywidgets absent : résultat pour les valeurs par défaut — changez-les dans l’appel pour explorer)')
        f(**{k: v[3] for k, v in curseurs.items()})
from sklearn import datasets
from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA, NMF
from sklearn.mixture import GaussianMixture
from sklearn.manifold import TSNE
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import linkage, dendrogram
import sklearn
print('Environnement prêt. scikit-learn', sklearn.__version__, '| graine =', SEED)

## 1 — Données non étiquetées et géométrie  *(diapos 19–20)*

Une matrice $X\in\mathbb{R}^{n\times p}$ contient $n$ observations et $p$ caractéristiques. En MLNS, les algorithmes utilisent $X$ **sans cible $y$**.

Distance euclidienne : $d_2(x,z)=\sqrt{\sum_{j}(x_j-z_j)^2}$. Distance de Manhattan : $d_1(x,z)=\sum_j|x_j-z_j|$.

**Calculs à la main.** Pour $x=(1,2)$ et $z=(4,6)$ : $d_2=\sqrt{3^2+4^2}=5$ et $d_1=3+4=7$. Exemple du cours : $A=(1,1)$, $B=(5,4)$ : $d_2=\sqrt{16+9}=5$.

In [ ]:
x, z = np.array([1., 2.]), np.array([4., 6.])
print('Euclidienne =', np.linalg.norm(x - z), '| Manhattan =', np.abs(x - z).sum())
assert np.isclose(np.linalg.norm(x - z), 5)

X_blob, y_cache = make_blobs(n_samples=180, centers=3, cluster_std=.75, random_state=SEED)   # y_cache ne sera regardé qu'à la fin
plt.scatter(*X_blob.T, s=14, color='gray'); plt.title('Des observations, aucune étiquette : combien de groupes voyez-vous ?'); plt.show()

### Pourquoi normaliser ?

Si une caractéristique varie de 0 à 100 000 et une autre de 0 à 10, la première domine les distances. **Standardisation :** $z_{ij}=(x_{ij}-\mu_j)/s_j$.

Ci-dessous, deux groupes se distinguent par l’**âge**, mais le **revenu**, exprimé dans une grande unité, écrase tout : k-means se trompe tant que l’on ne standardise pas.

In [ ]:
g = np.random.default_rng(SEED)
age = np.r_[g.normal(25, 3, 60), g.normal(55, 3, 60)]; revenu = g.normal(30000, 8000, 120)
X_ech = np.c_[age, revenu]; vrai = np.r_[np.zeros(60), np.ones(60)]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for a_, Xv, titre in [(ax[0], X_ech, 'Données brutes'), (ax[1], StandardScaler().fit_transform(X_ech), 'Données standardisées')]:
    lab = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit_predict(Xv)
    a_.scatter(X_ech[:, 0], X_ech[:, 1], c=lab, s=14, cmap='viridis'); a_.set_xlabel('Âge'); a_.set_ylabel('Revenu')
    a_.set_title(f'{titre}\naccord avec les deux tranches d’âge (ARI) = {adjusted_rand_score(vrai, lab):.2f}')
plt.tight_layout(); plt.show()

**À faire**

- 🟢 Calculez à la main la distance euclidienne entre $(0,0)$ et $(6,8)$.
- 🟡 Standardisez à la main la série $2, 4, 6$ (écart-type « population », `ddof=0`), puis vérifiez.
- 🔴 Dans quel cas la standardisation peut-elle **nuire** ? Construisez un exemple où une variable de pur bruit, une fois standardisée, détruit un regroupement évident.

In [ ]:
ma_distance = None                  # 🟢
verifier('d((0,0),(6,8))', ma_distance, 10)
ma_serie = None                     # 🟡 liste de trois valeurs
verifier('Standardisation de 2, 4, 6', ma_serie, StandardScaler().fit_transform([[2.], [4.], [6.]]).ravel())

## 2 — k-means : calcul manuel complet  *(diapos 21–22)*

Quatre points : $A=(1,1)$, $B=(1,2)$, $C=(5,5)$, $D=(6,5)$. On choisit $k=2$ et les centres initiaux $\mu_1=(1,1)$, $\mu_2=(6,5)$.

**Affectation :** $c_i=\arg\min_j\|x_i-\mu_j\|^2$.  **Mise à jour :** $\mu_j=\dfrac1{|C_j|}\sum_{i\in C_j}x_i$.

**À la main.** $B$ est à distance $1$ du centre 1 et $\sqrt{41}$ du centre 2 ; $C$ est à $\sqrt{32}$ du centre 1 et $1$ du centre 2. Donc $C_1=\{A,B\}$, $C_2=\{C,D\}$. Nouveaux centres : $(1\,;1{,}5)$ et $(5{,}5\,;5)$.

**Inertie :** $J=\sum_i\|x_i-\mu_{c_i}\|^2=0{,}25\times4=1$.

In [ ]:
P = np.array([[1., 1.], [1., 2.], [5., 5.], [6., 5.]]); centres = np.array([[1., 1.], [6., 5.]])
d2 = ((P[:, None, :] - centres[None, :, :]) ** 2).sum(axis=2); labels = d2.argmin(axis=1)
nouveaux = np.vstack([P[labels == j].mean(axis=0) for j in range(2)])
display(pd.DataFrame(d2, columns=['d² au centre 1', 'd² au centre 2'], index=list('ABCD')))
print('Groupes :', labels, '| nouveaux centres :', nouveaux.tolist(), '| inertie :', ((P - nouveaux[labels]) ** 2).sum())
assert np.isclose(((P - nouveaux[labels]) ** 2).sum(), 1)

### k-means programmé sans bibliothèque, itération par itération

Chaque itération de Lloyd alterne **affectation au centre le plus proche** et **recalcul des moyennes**. L’inertie ne peut pas augmenter, mais l’algorithme peut s’arrêter dans un minimum local qui dépend des centres de départ.

In [ ]:
def kmeans_maison(X, centres, iterations=10):
    C = centres.copy(); historique = []
    for _ in range(iterations):
        lab = ((X[:, None, :] - C[None, :, :]) ** 2).sum(2).argmin(1)                    # affectation
        historique.append((C.copy(), lab, ((X - C[lab]) ** 2).sum()))
        C = np.array([X[lab == j].mean(0) if (lab == j).any() else C[j] for j in range(len(C))])   # mise à jour
    return historique

hist = kmeans_maison(X_blob, X_blob[[0, 60, 120]], iterations=8)
fig, axes = plt.subplots(1, 4, figsize=(13, 3))
for t, ax in zip([0, 1, 2, 7], axes):
    C, lab, J = hist[t]
    ax.scatter(*X_blob.T, c=lab, s=12, cmap='viridis'); ax.scatter(*C.T, marker='X', s=150, color='red', edgecolor='black')
    ax.set_title(f'Itération {t} — inertie {J:.0f}'); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print('Inertie par itération :', [round(h[2]) for h in hist])

In [ ]:
def explorer_kmeans(k=3):
    m = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X_blob)
    plt.figure(figsize=(6, 4)); plt.scatter(*X_blob.T, c=m.labels_, s=18, cmap='viridis')
    plt.scatter(*m.cluster_centers_.T, s=200, marker='X', color='red', edgecolor='black'); plt.title(f'k = {k} | inertie = {m.inertia_:.1f}'); plt.show()

explorer(explorer_kmeans, k=(2, 7, 1, 3))

**À faire**

- 🟢 Dans l’explorateur, essayez $k=2$, $3$, $5$. Pour quelle valeur les groupes vous paraissent-ils « naturels » ?
- 🟡 Reprenez le calcul manuel avec les centres initiaux $\mu_1=(1,1)$ et $\mu_2=(1,2)$. Quels groupes obtient-on après une itération ? Et quelle inertie après la mise à jour ?
- 🔴 Démontrez que chacune des deux étapes de Lloyd ne peut pas augmenter $J$, puis expliquez pourquoi cela n’assure pas un minimum global. Lancez `kmeans_maison` avec 20 initialisations aléatoires et tracez l’histogramme des inerties finales.

In [ ]:
mon_inertie = None                  # 🟡 inertie après la première mise à jour (centres initiaux A et B)
h2 = kmeans_maison(P, P[[0, 1]], iterations=2)
verifier('Inertie après une mise à jour', mon_inertie, h2[1][2])

## 3 — Choisir $k$ : coude et silhouette  *(diapo 23)*

L’inertie décroît toujours quand $k$ augmente : **ne pas choisir le plus grand $k$**. On cherche le « coude ».

Pour une observation $i$ : $a(i)$ = distance moyenne aux points de son groupe ; $b(i)$ = plus petite distance moyenne à un autre groupe.

$$s(i)=\frac{b(i)-a(i)}{\max\{a(i),b(i)\}}\in[-1,1].$$

**À la main.** $a=2$, $b=5$ : $s=(5-2)/5=0{,}6$. $a=2$, $b=6$ : $s=4/6\approx0{,}667$.

In [ ]:
ks = range(2, 9); inertie, silhouettes = [], []
for k in ks:
    m = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X_blob)
    inertie.append(m.inertia_); silhouettes.append(silhouette_score(X_blob, m.labels_))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(list(ks), inertie, 'o-'); ax[0].set_title('Méthode du coude'); ax[0].set_xlabel('k'); ax[0].set_ylabel('inertie')
ax[1].plot(list(ks), silhouettes, 'o-'); ax[1].set_title('Silhouette moyenne'); ax[1].set_xlabel('k'); plt.tight_layout(); plt.show()

**À faire**

- 🟢 Où se situe le coude ? Quel $k$ maximise la silhouette ?
- 🟡 Calculez $s$ pour $a=3$, $b=4$, puis pour $a=4$, $b=3$. Comment interpréter une silhouette négative ?
- 🔴 Implémentez la silhouette moyenne sans scikit-learn et retrouvez `silhouette_score` pour $k=3$.

In [ ]:
ma_s1, ma_s2 = None, None           # 🟡
verifier('s pour a = 3, b = 4', ma_s1, (4 - 3) / 4); verifier('s pour a = 4, b = 3', ma_s2, (3 - 4) / 4)

## 4 — DBSCAN : densité, bruit et paramètres  *(diapo 24)*

DBSCAN utilise un rayon $\varepsilon$ et `min_samples`. Un point **cœur** possède au moins `min_samples` observations dans son voisinage $\varepsilon$ (lui-même inclus) : $|N_\varepsilon(x)|=5$ et MinPts $=4$ → cœur. Un point **bord** est atteignable depuis un cœur sans l’être lui-même. L’étiquette `-1` désigne le **bruit**.

La méthode retrouve des groupes non convexes — ce que k-means ne sait pas faire — mais elle est sensible à l’échelle et aux paramètres.

In [ ]:
X_moon, _ = make_moons(n_samples=350, noise=.07, random_state=SEED)
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
axes[0].scatter(*X_moon.T, c=KMeans(n_clusters=2, n_init=10, random_state=SEED).fit_predict(X_moon), s=10, cmap='tab10'); axes[0].set_title('k-means (k = 2) : échec')
for eps, ax in zip([.10, .20, .40], axes[1:]):
    m = DBSCAN(eps=eps, min_samples=5).fit(X_moon)
    ax.scatter(*X_moon.T, c=m.labels_, s=10, cmap='tab10')
    ax.set_title(f'DBSCAN ε = {eps} | {len(set(m.labels_)) - (-1 in m.labels_)} groupe(s), bruit = {(m.labels_ == -1).sum()}')
plt.tight_layout(); plt.show()

**À faire**

- 🟢 Pour quelle valeur de $\varepsilon$ retrouve-t-on les deux croissants ? Que se passe-t-il si $\varepsilon$ est trop petit ? trop grand ?
- 🟡 Un point possède 3 voisins (lui compris) dans son rayon $\varepsilon$ et `min_samples = 5`. Peut-il tout de même appartenir à un groupe ? À quelle condition ?
- 🔴 **Choisir $\varepsilon$.** Tracez, triées par ordre croissant, les distances de chaque point à son 5ᵉ plus proche voisin (`NearestNeighbors`). Où se trouve le coude ? Est-il cohérent avec vos essais ?

In [ ]:
# 🔴 point de départ : distances au 5ᵉ voisin
dist_k = np.sort(NearestNeighbors(n_neighbors=5).fit(X_moon).kneighbors(X_moon)[0][:, -1])
plt.plot(dist_k); plt.xlabel('points triés'); plt.ylabel('distance au 5ᵉ voisin'); plt.title('Graphique des k-distances'); plt.show()

## 5 — Classification hiérarchique et dendrogramme  *(diapos 25–26)*

La classification ascendante part de $n$ groupes d’un seul point et fusionne, à chaque étape, les deux groupes les plus proches. Le **dendrogramme** montre les fusions et leur hauteur ; une coupe horizontale donne autant de groupes que de branches traversées.

Les critères `single`, `complete`, `average` et `ward` ne mesurent pas la distance entre groupes de la même façon : la liaison simple prend $\min_{a\in A,b\in B}d(a,b)$ et peut « chaîner » ; Ward limite l’augmentation de variance intra-groupe.

In [ ]:
Z = linkage(X_blob[:25], method='ward')
plt.figure(figsize=(10, 3.8)); dendrogram(Z); plt.axhline(8, color='red', ls='--', label='exemple de coupe')
plt.title('Dendrogramme — liaison de Ward (25 observations)'); plt.ylabel('Hauteur de fusion'); plt.legend(); plt.show()
for methode in ['single', 'complete', 'ward']:
    lab = AgglomerativeClustering(n_clusters=3, linkage=methode).fit_predict(X_blob)
    print(f'{methode:9s} silhouette = {silhouette_score(X_blob, lab):.3f}')

**À faire**

- 🟡 Combien de groupes la coupe rouge produit-elle ? Où couper pour en obtenir deux ?
- 🟡 Cinq points sur une droite : $0$, $1$, $3$, $6$, $10$. Déroulez à la main la liaison simple : ordre et hauteurs des fusions.
- 🔴 Appliquez la liaison simple puis Ward aux deux croissants de la section 4. Laquelle réussit ? Expliquez par l’effet de chaînage.

In [ ]:
mes_hauteurs = None                 # 🟡 liste des 4 hauteurs de fusion, dans l'ordre
verifier('Hauteurs (liaison simple)', mes_hauteurs, linkage(np.array([[0.], [1.], [3.], [6.], [10.]]), method='single')[:, 2])

## 6 — Mélanges gaussiens (GMM) et responsabilités  *(diapo 27)*

Un mélange suppose $p(x)=\sum_{k=1}^K\pi_k\,\mathcal N(x\mid\mu_k,\Sigma_k)$, avec $\pi_k\ge0$ et $\sum_k\pi_k=1$.

**Étape E :** calculer les responsabilités $r_{ik}=P(z_i=k\mid x_i)$. **Étape M :** réestimer les paramètres pondérés. À la différence de k-means, l’affectation est **probabiliste** : un point situé entre deux groupes appartient partiellement à chacun.

In [ ]:
gmm = GaussianMixture(n_components=3, random_state=SEED).fit(X_blob)
resp = gmm.predict_proba(X_blob); assert np.allclose(resp.sum(axis=1), 1)
incertitude = 1 - resp.max(axis=1)
plt.scatter(*X_blob.T, c=incertitude, s=20, cmap='magma'); plt.colorbar(label='incertitude = 1 − responsabilité maximale')
plt.scatter(*gmm.means_.T, marker='X', s=180, color='cyan', edgecolor='black'); plt.title('GMM : les points les plus clairs sont partagés entre groupes'); plt.show()
print('Les 3 points les plus partagés :'); display(pd.DataFrame(resp[np.argsort(-incertitude)[:3]], columns=['G1', 'G2', 'G3']).round(3))

**À faire**

- 🟡 Deux gaussiennes de même variance et de même poids sont centrées en 0 et en 4. Quelles sont les responsabilités d’un point situé en $x=2$ ?
- 🔴 Calculez à la main, puis vérifiez, la responsabilité de la première composante pour $x=1$ ($\sigma=1$, $\pi_1=\pi_2=0{,}5$).
- 🔴 Comparez les critères BIC pour $K=1$ à $6$ (`gmm.bic(X)`). Le nombre de composantes retenu a-t-il un sens ici ? Et sur les deux croissants ?

In [ ]:
ma_responsabilite = None            # 🔴 r₁ pour x = 1
dens = lambda v, mu: np.exp(-.5 * (v - mu) ** 2)
verifier('r₁(x = 1)', ma_responsabilite, dens(1, 0) / (dens(1, 0) + dens(1, 4)), tol=1e-3)

## 7 — ACP : la démarche mathématique  *(diapos 28–30)*

**But :** trouver, sans étiquettes, les directions orthogonales de variance maximale.

1. Centrer : $X_c=X-\mathbf 1\bar x^\top$.
2. Covariance empirique : $S=\dfrac1{n-1}X_c^\top X_c$.
3. Résoudre $Sv_j=\lambda_jv_j$.
4. Projeter : $Z=X_cV_q$.
5. Part de variance de l’axe $j$ : $\lambda_j/\sum_\ell\lambda_\ell$.

**À la main.** Centrer $1,3,5$ : moyenne 3, valeurs centrées $(-2,0,2)$. Points $(1,1),(2,2),(3,3)$ : moyenne $(2,2)$, covariance $\begin{pmatrix}1&1\\1&1\end{pmatrix}$, valeurs propres $2$ et $0$, premier axe $(1,1)/\sqrt2$ : 100 % de la variance sur PC1.

In [ ]:
Xp = np.array([[1., 1.], [2., 2.], [3., 3.]])
Xc = Xp - Xp.mean(0); S = Xc.T @ Xc / (len(Xp) - 1)
lam, V = np.linalg.eigh(S); ordre = np.argsort(lam)[::-1]; lam, V = lam[ordre], V[:, ordre]
print('Moyenne', Xp.mean(0), '\nCovariance\n', S, '\nValeurs propres', lam.round(6), '\nProjections sur PC1', np.round(Xc @ V[:, 0], 3))
assert np.isclose(PCA(n_components=1).fit(Xp).explained_variance_ratio_[0], 1)

### ACP sur *Wine* (jeu réel intégré)

Les 13 variables ont des unités différentes : on standardise avant l’ACP. Les classes du jeu ne servent qu’à **colorer** la figure de droite, après la transformation.

In [ ]:
wine = datasets.load_wine(); X_w = StandardScaler().fit_transform(wine.data)
pca = PCA().fit(X_w); Zw = pca.transform(X_w)[:, :2]; cumul = np.cumsum(pca.explained_variance_ratio_)
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].bar(range(1, 14), pca.explained_variance_ratio_); ax[0].plot(range(1, 14), cumul, 'o-', color='green'); ax[0].axhline(.8, ls='--', color='red')
ax[0].set_xlabel('composante'); ax[0].set_title(f'Variance expliquée — 2 axes : {cumul[1]:.0%}')
ax[1].scatter(*Zw.T, s=18, color='gray'); ax[1].set_title('ACP sans étiquettes')
ax[2].scatter(*Zw.T, c=wine.target, s=18, cmap='viridis'); ax[2].set_title('Couleurs = classes connues (a posteriori)')
plt.tight_layout(); plt.show()
print('Composantes nécessaires pour atteindre 80 % :', int(np.argmax(cumul >= .8)) + 1)

**À faire**

- 🟢 Combien de composantes faut-il pour conserver 80 % de la variance ? Quelle part perd-on en n’en gardant que deux ?
- 🟡 Avec les parts 58 %, 25 %, 11 %, 6 % (exemple du cours), combien d’axes pour dépasser 90 % ?
- 🟡 Calculez à la main la matrice de covariance des points $(0,0)$, $(2,0)$, $(1,3)$ … puis vérifiez.
- 🔴 Refaites l’ACP de *Wine* **sans standardiser**. Quelle variable domine PC1 ? Pourquoi ? Retrouvez ensuite `pca.components_` par décomposition en valeurs singulières de $X_c$.

In [ ]:
mes_axes = None                     # 🟡 nombre d'axes pour dépasser 90 % avec (58, 25, 11, 6)
verifier('Nombre d’axes', mes_axes, int(np.argmax(np.cumsum([58, 25, 11, 6]) > 90)) + 1, tol=.1)
ma_covariance = None                # 🟡 matrice 2×2 sous forme de liste de listes
verifier('Covariance', ma_covariance, np.cov(np.array([[0., 0.], [2., 0.], [1., 3.]]).T))

## 8 — t-SNE : visualiser, pas prouver  *(diapo 31)*

t-SNE préserve surtout les **voisinages locaux**. Les distances entre amas et leur taille apparente ne sont pas des mesures fiables de séparation globale ; la carte change avec la perplexité et la graine.

In [ ]:
digits = datasets.load_digits(); X_d = StandardScaler().fit_transform(digits.data[:400]); y_d = digits.target[:400]
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].scatter(*PCA(n_components=2).fit_transform(X_d).T, c=y_d, s=9, cmap='tab10'); ax[0].set_title('ACP')
for a_, graine in zip(ax[1:], [0, 1]):
    emb = TSNE(n_components=2, perplexity=25, init='pca', random_state=graine).fit_transform(X_d)
    a_.scatter(*emb.T, c=y_d, s=9, cmap='tab10'); a_.set_title(f't-SNE — graine {graine} (couleurs a posteriori)')
plt.tight_layout(); plt.show()

**À faire**

- 🟢 Les deux cartes t-SNE sont-elles identiques ? Qu’est-ce qui reste stable, qu’est-ce qui change ?
- 🟡 Essayez `perplexity = 5` puis `50`. Peut-on conclure, d’après la carte, que deux chiffres sont « proches » ?
- 🔴 Mesurez la préservation des voisinages : pour chaque point, quelle fraction de ses 10 plus proches voisins d’origine se retrouve parmi ses 10 voisins sur la carte ? Comparez ACP et t-SNE.

## 9 — Détection d’anomalies  *(diapo 32)*

Une anomalie est un point atypique **relativement au modèle et au jeu de données** ; ce n’est pas nécessairement une fraude ou une erreur. On calcule un score et on signale ce qui dépasse un seuil $\tau$ : $\text{score}(x)>\tau\Rightarrow$ anomalie présumée.

In [ ]:
g = np.random.default_rng(SEED)
X_an = np.vstack([g.normal(size=(200, 2)), [[5, 5], [-5, 4], [4, -5]]])
score_dist = np.linalg.norm(X_an - X_an.mean(0), axis=1)                 # score simple : distance au centre

def voir_seuil(tau=3.0):
    alerte = score_dist > tau
    plt.scatter(*X_an.T, c=alerte, cmap='coolwarm', s=18); plt.gca().add_patch(plt.Circle(X_an.mean(0), tau, fill=False, ls='--'))
    plt.axis('equal'); plt.title(f'τ = {tau:.1f} → {alerte.sum()} alerte(s) sur {len(X_an)}'); plt.show()

explorer(voir_seuil, tau=(1.0, 6.0, 0.5, 3.0))
flags = IsolationForest(contamination=.02, random_state=SEED).fit_predict(X_an)
print('IsolationForest (contamination = 2 %) signale', (flags == -1).sum(), 'points')

**À faire**

- 🟢 Que devient le nombre d’alertes quand le seuil augmente ? Les trois points ajoutés sont-ils toujours signalés ?
- 🟡 Avec $\tau=2$, combien de points « normaux » sont signalés à tort ? Quel est le coût d’un faux positif pour une banque ? pour un hôpital ?
- 🔴 Remplacez la distance euclidienne au centre par la distance de Mahalanobis sur un nuage **allongé**. Pourquoi est-elle mieux adaptée ?

## 10 — Règles d’association : calculs manuels  *(diapo 33)*

$$\text{support}(A\Rightarrow B)=P(A\cap B),\qquad \text{confiance}=\frac{P(A\cap B)}{P(A)},\qquad \text{lift}=\frac{\text{confiance}}{P(B)}.$$

**Exemple du cours.** Sur 10 paniers : pain 6, lait 5, les deux 4. Support $=0{,}4$ ; confiance $=4/6\approx0{,}667$ ; lift $=0{,}667/0{,}5\approx1{,}333$.

**Second exemple.** T1 = {pain, lait}, T2 = {pain, lait, beurre}, T3 = {pain, beurre}, T4 = {lait, beurre}. Pour *pain ⇒ lait* : support $=2/4$ ; confiance $=(2/4)/(3/4)=2/3$ ; lift $=(2/3)/(3/4)=8/9\approx0{,}889$.

> Un lift supérieur à 1 signale une cooccurrence plus fréquente que sous l’indépendance. **Une association n’est pas une causalité.**

In [ ]:
def regles(transactions, A, B):
    n = len(transactions); sup = lambda items: sum(set(items) <= t for t in transactions) / n
    s = sup(A | B); conf = s / sup(A); return s, conf, conf / sup(B)

T4 = [{'pain', 'lait'}, {'pain', 'lait', 'beurre'}, {'pain', 'beurre'}, {'lait', 'beurre'}]
s, c, l = regles(T4, {'pain'}, {'lait'}); print(f'4 paniers  : support = {s:.3f}, confiance = {c:.3f}, lift = {l:.3f}'); assert np.isclose(l, 8 / 9)
T10 = [{'pain', 'lait'}] * 4 + [{'pain'}] * 2 + [{'lait'}] + [set()] * 3
s, c, l = regles(T10, {'pain'}, {'lait'}); print(f'10 paniers : support = {s:.3f}, confiance = {c:.3f}, lift = {l:.3f}'); assert np.isclose(l, 4 / 3)

**À faire**

- 🟢 Dans le second exemple, combien de paniers contiennent à la fois du lait et du beurre ?
- 🟡 Calculez à la main support, confiance et lift de *lait ⇒ beurre* (4 paniers), puis vérifiez.
- 🔴 Écrivez une fonction qui énumère toutes les règles $A\Rightarrow B$ à un article de chaque côté, avec un support minimal de 0,25, triées par lift. Pourquoi l’algorithme Apriori évite-t-il d’examiner tous les sous-ensembles ?

In [ ]:
mon_support, ma_confiance, mon_lift = None, None, None      # 🟡 pour lait ⇒ beurre
s, c, l = regles(T4, {'lait'}, {'beurre'})
verifier('support', mon_support, s); verifier('confiance', ma_confiance, c); verifier('lift', mon_lift, l)

## 11 — NMF : factorisation non négative  *(diapo 34)*

Pour $X\ge0$, on cherche $W\ge0$ et $H\ge0$ tels que $X\approx WH$. Avec une matrice **documents × mots**, $H$ (thèmes × mots) décrit des thèmes et $W$ (documents × thèmes) le poids de chaque thème dans chaque document.

Le mini-corpus ci-dessous mélange deux sujets ; on ne le dit pas à l’algorithme.

In [ ]:
mots = ['but', 'match', 'équipe', 'vote', 'loi', 'ministre']
X_docs = np.array([[3, 2, 3, 0, 0, 1], [2, 3, 2, 0, 1, 0], [0, 0, 1, 3, 2, 3], [0, 1, 0, 2, 3, 2], [3, 3, 2, 0, 0, 0], [0, 0, 0, 3, 3, 2]], dtype=float)
nmf = NMF(n_components=2, init='nndsvda', max_iter=2000, random_state=SEED)
W = nmf.fit_transform(X_docs); H = nmf.components_
display(pd.DataFrame(H, columns=mots, index=['thème 1', 'thème 2']).round(2))
display(pd.DataFrame(W, columns=['thème 1', 'thème 2'], index=[f'doc {i + 1}' for i in range(6)]).round(2))
print('Erreur de reconstruction ‖X − WH‖ =', round(np.linalg.norm(X_docs - W @ H), 3))

**À faire**

- 🟢 Quels mots caractérisent chaque thème ? Quels documents parlent surtout de sport ?
- 🟡 Combien de coefficients contiennent $W$ et $H$ réunis, contre combien de valeurs dans $X$ ? Que devient l’erreur avec `n_components = 1` ? `3` ?
- 🔴 Programmez les mises à jour multiplicatives $H\leftarrow H\circ\dfrac{W^\top X}{W^\top WH}$, $W\leftarrow W\circ\dfrac{XH^\top}{WHH^\top}$ et tracez l’erreur au fil des itérations. Les facteurs obtenus sont-ils uniques ?

## 12 — Évaluer un clustering : interne, externe, stabilité  *(diapo 35)*

- **Interne :** silhouette, Davies–Bouldin, Calinski–Harabasz.
- **Externe :** ARI ou NMI, seulement si des étiquettes de référence existent — et jamais pour choisir le modèle.
- **Stabilité :** comparer plusieurs initialisations, sous-échantillons et hyperparamètres.
- **Interprétabilité :** profils des groupes, tailles, cas limites, avis d’un expert du domaine.

> Une silhouette élevée ne garantit pas une vérité métier.

In [ ]:
iris = datasets.load_iris(); Xi = StandardScaler().fit_transform(iris.data); lignes = []
for k in range(2, 7):
    m = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Xi)
    # stabilité : accord moyen (ARI) entre 5 exécutions à une seule initialisation
    runs = [KMeans(n_clusters=k, n_init=1, random_state=s).fit_predict(Xi) for s in range(5)]
    stab = np.mean([adjusted_rand_score(runs[i], runs[j]) for i in range(5) for j in range(i + 1, 5)])
    lignes.append({'k': k, 'silhouette': silhouette_score(Xi, m.labels_), 'stabilité (ARI entre exécutions)': stab, 'ARI vs espèces (a posteriori)': adjusted_rand_score(iris.target, m.labels_)})
display(pd.DataFrame(lignes).round(3))

**À faire**

- 🟡 La silhouette désigne $k=2$, alors que le jeu contient 3 espèces. Qui a « raison » ? Que nous apprend ce désaccord ?
- 🔴 Ajoutez une mesure de stabilité par **sous-échantillonnage** (80 % des observations, 10 répétitions). Quel $k$ est le plus stable ?
- 🔴 Comment comparer deux regroupements quand il n’existe aucune vérité terrain ?

## 13 — Défis et mini-projets (à rendre)

| Niveau | Sujet | Travail demandé |
|---|---|---|
| 🟢 L1–L2 | **Défi découverte — regrouper des fleurs** | Appliquer k-means ($k=2,3,4$) à `load_iris()` standardisé, afficher les groupes sur les deux premières composantes d’ACP, décrire chaque groupe en une phrase. |
| 🟡 L3 | **Défi — k-means contre DBSCAN** | Sur `make_moons` et `make_blobs`, comparer les deux méthodes, choisir $\varepsilon$ par le graphique des k-distances, expliquer quand chaque méthode échoue. |
| 🔴 M1–M2 | **Mini-projets A et B** ci-dessous | Protocole complet, sans fuite d’étiquettes. |

### Mini-projet A — Segmentation non supervisée 🔴

**Jeu :** `load_wine()` (178 observations, 13 caractéristiques). Les catégories d’origine sont **masquées pendant l’apprentissage**.

1. Dictionnaire de données, distributions, valeurs manquantes.
2. Standardiser les 13 variables ; expliquer pourquoi.
3. Comparer k-means ($k=2\ldots8$), DBSCAN (grille de $\varepsilon$ et `min_samples`) et classification hiérarchique.
4. Pour chaque méthode : nombre de groupes, tailles, bruit, silhouette lorsqu’elle est définie.
5. Visualiser en 2D par ACP, sans prétendre que la projection valide le regroupement.
6. Décrire les profils moyens des groupes et leurs limites.
7. **Après** le choix du modèle, calculer l’ARI par rapport aux catégories connues et discuter.
8. Conclusion argumentée : quel algorithme, et pourquoi ?

In [ ]:
# MINI-PROJET A — à compléter
wine_df = datasets.load_wine(as_frame=True); X_A = wine_df.data.copy()
# TODO A1 : exploration, standardisation
# TODO A2 : k-means, DBSCAN, hiérarchique
# TODO A3 : tableau comparatif et silhouettes (si définies)
# TODO A4 : visualisation ACP et profils de groupes
# TODO A5 : évaluation externe uniquement après la sélection
print('Projet A :', X_A.shape, '— à compléter')

### Mini-projet B — Réduction de dimension et découverte de structure 🔴

**Jeu :** `load_digits()` (1 797 images de chiffres manuscrits, 64 intensités de pixels).

1. Visualiser 12 images ; examiner la dimension et les valeurs des pixels.
2. ACP sans étiquettes ; variance expliquée cumulée ; plus petit nombre de composantes couvrant 90 %.
3. Reconstruire 6 images avec $q=5,15,30$ composantes ; comparer à l’œil et par erreur quadratique moyenne.
4. Comparer k-means sur les pixels et sur les 20 premières composantes.
5. Visualiser ACP 2D et t-SNE 2D ; discuter les risques d’interprétation.
6. Analyse de stabilité sur au moins trois graines.
7. Ne révéler les étiquettes qu’à la fin, pour calculer l’ARI.

**Livrables (A et B) :** notebook reproductible + synthèse d’une à deux pages + trois figures légendées.

In [ ]:
# MINI-PROJET B — à compléter
X_B = datasets.load_digits().data.copy()      # pas d'étiquettes pour ajuster les modèles
# TODO B1 : visualiser les images
# TODO B2 : ACP + variance expliquée cumulée
# TODO B3 : reconstruction et erreur quadratique moyenne
# TODO B4 : k-means sur pixels et sur composantes
# TODO B5 : ACP / t-SNE et stabilité
# TODO B6 : ARI uniquement après les choix non supervisés
print('Projet B :', X_B.shape, '— à compléter')

## 14 — Barème (100 points)

| Critère | Points |
|---|---:|
| Compréhension mathématique et justifications | 20 |
| Préparation des données et reproductibilité | 15 |
| Expérimentation méthodique et comparaison | 20 |
| Visualisations et interprétation | 15 |
| Évaluation, stabilité et limites | 15 |
| Qualité du code, clarté et conclusion | 15 |

**Règles :** pas de fuite d’étiquettes ; graine fixée ; figures avec axes et légendes ; chaque métrique interprétée ; distinction nette entre observation et conclusion. Pour les parcours 🟢 et 🟡, le barème s’applique aux exercices du niveau choisi et au défi correspondant.

## 15 — Autoévaluation

1. Pourquoi la normalisation modifie-t-elle les groupes obtenus par k-means ?
2. Déroulez une itération complète de Lloyd à la main.
3. Pourquoi l’inertie ne suffit-elle pas à choisir $k$ ?
4. Comment interpréter une silhouette négative ?
5. Que signifie l’étiquette `-1` de DBSCAN ?
6. Quelle différence entre affectation dure (k-means) et probabiliste (GMM) ?
7. Calculez une covariance $2\times2$ et une projection d’ACP.
8. Pourquoi une carte t-SNE ne prouve-t-elle pas l’existence de classes naturelles ?
9. Quelle différence entre une anomalie statistique et une fraude avérée ?
10. Pourquoi un lift élevé ne prouve-t-il pas une relation causale ?
11. Quand peut-on calculer un ARI sans compromettre le protocole ?
12. Comment comparer deux regroupements sans vérité terrain ?

**Conclusion.** Le MLNS explore des structures ; les résultats dépendent des données, de la représentation, de la distance, des hypothèses et des hyperparamètres. Une partition n’est pas automatiquement une vérité scientifique. En pratique, il prépare souvent des caractéristiques pour un modèle supervisé.

**Suite :** TP 4 — Deep Learning et Transformers.